In [1]:
from pyscf import gto, fci, mcscf
import numpy as np
from gpu4pyscf import scf


/home/mert/.local/lib/python3.10/site-packages/gpu4pyscf/lib/cutensor.py:154: UserWarning: using cupy as the tensor contraction engine.
  warnings.warn(f'using {contract_engine} as the tensor contraction engine.')


In [2]:
# Molecular Information part
# !!!!! in CI part mo not forget the change electrons 
# !!!!! also orbitals are important  

xyz = '''
C   0.000   1.201   0.000
C   1.142   0.371   0.000
C   0.706  -0.972   0.000
C  -1.142   0.371   0.000
C  -0.706  -0.972   0.000
H   0.000   2.284   0.000
H   2.172   0.706   0.000
H   1.343  -1.848   0.000
H  -2.172   0.706   0.000
H  -1.343  -1.848   0.000'''
molecule = 'C5H5_an'
spin = 0
charge =  -1


In [9]:
def rhf_calculation(xyz, molecule, spin, charge):
    mol = gto.M(
          atom= xyz,
          basis = 'sto-6g',
          charge = charge,
          verbose = 5 ,
          spin = spin, 
          symmetry = True,
          output  ="./"+molecule+".txt")

    mf = mol.RHF().run()
    mf.analyze() 
    S_ij = mf.get_ovlp()
    C_ij = mf.mo_coeff
    np.savetxt("C_"+molecule+'mo_coeff.txt', C_ij,fmt = '%.8f')
    np.savetxt('S_'+molecule+'_overlap.txt', S_ij,fmt = '%.8f')

    return mf, mol 

In [10]:
def orb_occ_table(orb_energy,orb_occ):
    Homo = None
    Lumo = None
    print ("orbital  energy   occ")

    for i in range (0,(len(orb_occ))):
        print(i ,"   ", orb_energy[i],"  ",orb_occ[i])
              
    for i in range(len(orb_occ) - 1):
        if orb_occ[i] == 2  and orb_occ[i+1] == 0:
            homo = i
            lumo = i + 1
            print(f"Homo_index: {homo}, Lumo_index: {lumo}")
    return homo, lumo


In [11]:
mf, mol  = rhf_calculation( xyz , molecule, spin, charge )
homo , lumo =orb_occ_table(mf.mo_energy, mf.mo_occ)

overwrite output file: ./C5H5_an.txt
orbital  energy   occ
0     -10.844557396216967    2.0
1     -10.844313952011724    2.0
2     -10.844301448619488    2.0
3     -10.843620863603638    2.0
4     -10.843613686520865    2.0
5     -0.7987156285468424    2.0
6     -0.613435262982289    2.0
7     -0.6133552962985276    2.0
8     -0.4199006460367889    2.0
9     -0.4198262217073183    2.0
10     -0.3837303797565601    2.0
11     -0.21303602051103843    2.0
12     -0.21283848680680673    2.0
13     -0.1835464981747834    2.0
14     -0.18353095013622966    2.0
15     -0.18189473067422746    2.0
16     0.04959115132455416    2.0
17     0.049668039771226    2.0
18     0.6489743311066373    0.0
19     0.6490937971595467    0.0
20     0.8484893360915149    0.0
21     0.9593932818091234    0.0
22     0.9596360220494711    0.0
23     0.9765011607506466    0.0
24     0.9765409298076788    0.0
25     1.0596969497053914    0.0
26     1.0598244598909812    0.0
27     1.2473666908225653    0.0
28     1

WARN: System has nonzero charge -1; the dipole moment is origin-dependent.
Location of origin: [0. 0. 0.]


In [12]:
n_elec= 6
n_orb = 5 # number of orbital can be determined based on the number of pi  orbitals 
casscf = mcscf.CASSCF(mf, n_orb, n_elec)
'''output in order
e_tot
e_cas
Fci vecs 
mo_coeff
mo_energy'''
mo = casscf.sort_mo([16,17,18,19,20])# the numbers are from the index of pi orbitals 
casscf.kernel(mo) 


(np.float64(-191.67123233997964),
 np.float64(-5.013662485441216),
 FCIvector([[-9.57976904e-01, -7.77896532e-06,  0.00000000e+00,
              0.00000000e+00,  0.00000000e+00,  1.82569979e-05,
              3.77376458e-06,  5.57948181e-02, -1.40780408e-05,
              0.00000000e+00],
            [-7.77896532e-06,  9.88728809e-02,  0.00000000e+00,
              0.00000000e+00,  0.00000000e+00,  2.79180086e-02,
              5.87210508e-02, -4.77872424e-06, -2.13745692e-03,
              0.00000000e+00],
            [ 0.00000000e+00,  0.00000000e+00,  9.88656154e-02,
              5.87213375e-02, -2.78768372e-02,  0.00000000e+00,
              0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
              2.13600538e-03],
            [ 0.00000000e+00,  0.00000000e+00,  5.87213375e-02,
              4.95614932e-02,  5.87351303e-02,  0.00000000e+00,
              0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
              1.67980584e-08],
            [ 0.00000000e+00,  0.00000000

In [13]:
casscf.fcisolver.max_cycle = 3000
casscf.canonicalize(mo_coeff=None, ci=None, eris=None, sort=False, cas_natorb=False, casdm1=None, verbose=3, with_meta_lowdin=False);

filename = f"{molecule}_Ci_coeffs.txt"

with open(filename, 'w', encoding='utf-8') as f:
    
    f.write("det-alpha,     det-beta,    CI coefficients\n")
    
    
    for c, ia, ib in casscf.fcisolver.large_ci(casscf.ci, n_orb, n_elec, tol=.000000001, return_strs=False):
        # Satır formatını oluşturalım
        line = f" {str(ia):<12} {str(ib):<12} {c:.15f}\n"
        
        #write in file 
        f.write(line)
        #show in terminal 
        print(line.strip())

[0 1 2]      [0 1 2]      -0.957976904000284
[0 1 2]      [0 1 3]      -0.000007778965320
[0 1 2]      [0 2 4]      0.000018256997925
[0 1 2]      [1 2 4]      0.000003773764577
[0 1 2]      [0 3 4]      0.055794818078374
[0 1 2]      [1 3 4]      -0.000014078040820
[0 1 3]      [0 1 2]      -0.000007778965320
[0 1 3]      [0 1 3]      0.098872880879376
[0 1 3]      [0 2 4]      0.027918008552961
[0 1 3]      [1 2 4]      0.058721050798540
[0 1 3]      [0 3 4]      -0.000004778724235
[0 1 3]      [1 3 4]      -0.002137456921423
[0 2 3]      [0 2 3]      0.098865615445464
[0 2 3]      [1 2 3]      0.058721337463715
[0 2 3]      [0 1 4]      -0.027876837173594
[0 2 3]      [2 3 4]      0.002136005377600
[1 2 3]      [0 2 3]      0.058721337463715
[1 2 3]      [1 2 3]      0.049561493223652
[1 2 3]      [0 1 4]      0.058735130304389
[1 2 3]      [2 3 4]      0.000000016798058
[0 1 4]      [0 2 3]      -0.027876837173594
[0 1 4]      [1 2 3]      0.058735130304389
[0 1 4]      [0 1 4]    